In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;


In [0]:
%sql
-- Who has opened at least one email?
SELECT DISTINCT email
FROM bronze.raw_email_events
WHERE event_type = 'Opened Email'

email
jane.smith@gmail.com
sarah.williams@gmail.com
lisa.taylor@gmail.com
james.wilson@gmail.com
tom.evans@gmail.com
amy.robinson@gmail.com
helen.hall@gmail.com
kate.young@gmail.com


In [0]:
CREATE OR REPLACE TABLE silver.dim_customers_with_email_status AS

SELECT
    c.customer_id,
    c.email,
    c.first_name,
    c.last_name,
    c.total_spent,
    c.orders_count,
    c.email_consent,
    c.created_at,
    CASE
        WHEN o.email IS NOT NULL THEN true
        ELSE false
    END AS has_opened_email

FROM bronze.raw_customers c

LEFT JOIN (
    SELECT DISTINCT email
    FROM bronze.raw_email_events
    WHERE event_type = 'Opened Email'
) o ON lower(c.email) = lower(o.email)

num_affected_rows,num_inserted_rows


In [0]:
SELECT 
    email,
    first_name,
    has_opened_email
FROM silver.dim_customers_with_email_status
ORDER BY has_opened_email DESC

email,first_name,has_opened_email
amy.robinson@gmail.com,Amy,true
jane.smith@gmail.com,Jane,true
kate.young@gmail.com,Kate,true
james.wilson@gmail.com,James,true
sarah.williams@gmail.com,Sarah,true
helen.hall@gmail.com,Helen,true
lisa.taylor@gmail.com,Lisa,true
tom.evans@gmail.com,Tom,true
david.clark@gmail.com,David,false
mike.brown@gmail.com,Mike,false


In [0]:
CREATE DATABASE IF NOT EXISTS gold;

In [0]:
CREATE OR REPLACE TABLE gold.mart_dm_target_audience AS

SELECT
    customer_id,
    email,
    first_name,
    last_name,
    total_spent,
    orders_count,
    current_timestamp() AS list_generated_at

FROM silver.dim_customers_with_email_status

WHERE has_opened_email = false
  AND orders_count >= 1
  AND email_consent = 'subscribed'

ORDER BY total_spent DESC

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM gold.mart_dm_target_audience

customer_id,email,first_name,last_name,total_spent,orders_count,list_generated_at
C018,lucy.scott@gmail.com,Lucy,Scott,210.0,2,2026-03-30T14:33:38.365Z
C008,sophie.davies@gmail.com,Sophie,Davies,180.0,2,2026-03-30T14:33:38.365Z
C011,david.clark@gmail.com,David,Clark,155.0,2,2026-03-30T14:33:38.365Z
C019,john.green@gmail.com,John,Green,145.0,2,2026-03-30T14:33:38.365Z
C002,bob.jones@gmail.com,Bob,Jones,120.0,1,2026-03-30T14:33:38.365Z
C015,paul.allen@gmail.com,Paul,Allen,110.0,1,2026-03-30T14:33:38.365Z
C004,mike.brown@gmail.com,Mike,Brown,95.0,1,2026-03-30T14:33:38.365Z
C010,rachel.thomas@gmail.com,Rachel,Thomas,90.0,1,2026-03-30T14:33:38.365Z
C017,mark.king@gmail.com,Mark,King,85.0,1,2026-03-30T14:33:38.365Z
C006,emma.johnson@gmail.com,Emma,Johnson,75.0,1,2026-03-30T14:33:38.365Z
